# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SajidurCodes/flyrank-ml-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [6]:
import os
from pathlib import Path
from dotenv import load_dotenv
import duckdb
import pandas as pd

PROJECT_ROOT = Path(r"C:\Projects AI ML\flyrank-ml-starter")
load_dotenv(PROJECT_ROOT / ".env", override=True)
HF_TOKEN = os.getenv("HF_TOKEN")

con = duckdb.connect()
con.execute("CREATE SECRET hf_token (TYPE huggingface, TOKEN ?)", [HF_TOKEN])

WAREHOUSE = "hf://datasets/FlyRank/internship-warehouse"
FACT_MARCH = f"{WAREHOUSE}/fact_content_daily_performance/month=2026-03/*.parquet"
FACT_FEB   = f"{WAREHOUSE}/fact_content_daily_performance/month=2026-02/*.parquet"  # prior month, still past — safe to use
DIM_CONTENT = f"{WAREHOUSE}/dim_content.parquet"
DIM_CLIENTS = f"{WAREHOUSE}/dim_clients.parquet"

# TODO: confirm this still runs clean before anything else — if month=2026-02
# doesn't exist as a partition, swap to whatever the earliest safe prior month is.
print(con.sql(f"SELECT COUNT(*) FROM read_parquet('{FACT_MARCH}')").df())


   count_star()
0       9841378


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**The rule, in plain words:** rank pages for review primarily by how far their click-through rate falls short of what's normal for their search-position bucket, with staleness as a light secondary tiebreaker. Staleness turned out to be a much weaker signal than expected once tested (see verdicts below) — the rule leans on CTR-gap, not staleness, as the dominant driver.

**Two signals checked first, each with a bucket table (n printed) and a verdict:**

1. **Staleness** (`content_updated_date` from `dim_content`, clamped to `<= 2026-03-31` so no future edits leak into a March decision). This is the signal directly behind FlyRank's real refresh flag from the session.
   - Bucket table (quartiles by staleness, n=9,305/9,305/9,305/9,304 after the leakage clamp):
     - Q1 (freshest, ~33 days stale): avg click delta Feb→Mar = **+0.1**
     - Q2 (~34 days): **+0.2**
     - Q3 (~34 days): **-0.0**
     - Q4 (stalest, ~155 days): **+0.5**
   - **Verdict: MIXED, leaning FALSE.** Not monotonic — the stalest quartile actually shows the *best* trend, the opposite of what the refresh flag assumes. Magnitudes are also tiny (all within ±0.5 clicks), an order of magnitude smaller than the CTR-gap effect below. Staleness gets a low-to-zero weight in the rule as a result.
   - **Data-quality note that shaped this verdict:** the unclamped version of this query (before the leakage fix) produced *negative* `avg_days_stale` values — `dim_content` is a single unpartitioned snapshot as of the dataset's July 2026 export date, so most rows' `content_updated_date` falls after March 2026. Clamping to `<= 2026-03-31` is necessary to avoid leaking future edits into a March-decision feature, but it also drops the usable row count from ~73,500 to ~9,300 per quartile (~87% of joined rows excluded). That's a real limitation of this signal, not a bug: only a small, possibly unrepresentative slice of content has update history that's actually knowable as of March.

2. **CTR-vs-position gap** (`gsc_clicks` / `gsc_impressions` vs. an expected-CTR-by-position curve). This is the signal behind the CTR-fix logic from the session.
   - Bucket table:
     - `meeting_or_above` (n=23,948): avg click delta Feb→Mar = **+6.1**
     - `underperforming` (n=137,591): avg click delta Feb→Mar = **+0.4**
   - **Verdict: CONFIRMED.** Pages underperforming their expected CTR grew far less month-over-month than pages meeting/beating it — the direction the CTR-fix flag assumes, and the effect size is an order of magnitude larger than staleness. Note the bucket sizes are heavily imbalanced (137,591 vs 23,948); expected by construction since the benchmark is a mean and skewed distributions put more pages below the mean than above it — worth a caveat, not a disqualifier.

**Reason codes this rule can output** (single reason code per row, whichever signal contributed more to the score):
- `CTR_GAP_ONLY` — CTR gap drove the score, staleness didn't meaningfully contribute (expected to be the most common code given the verdicts above)
- `STALE_LOW_CTR` — both signals fired (rare, given staleness' weak effect)
- `STALE_ONLY` — staleness alone drove the score (should be uncommon/near-zero given the FALSE-leaning verdict)
- `HEALTHY` — neither signal fires; score stays low

**Action labels:**
- `REFRESH` — top-scoring pages, `STALE_LOW_CTR` or `STALE_ONLY`
- `CTR_FIX` — top-scoring pages, `CTR_GAP_ONLY`
- `MONITOR` — mid-range score, no immediate action
- `PROTECT` — low score, already healthy, don't touch it

In [7]:
content_schema = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{DIM_CONTENT}') LIMIT 0").df()
print(content_schema)

staleness = con.sql(f"""
    WITH march AS (
        SELECT client_hash_id, content_hash_id, SUM(gsc_clicks) AS clicks_march
        FROM read_parquet('{FACT_MARCH}')
        GROUP BY client_hash_id, content_hash_id
    ),
    feb AS (
        SELECT client_hash_id, content_hash_id, SUM(gsc_clicks) AS clicks_feb
        FROM read_parquet('{FACT_FEB}')
        GROUP BY client_hash_id, content_hash_id
    ),
    joined AS (
        SELECT
            m.client_hash_id, m.content_hash_id, m.clicks_march, f.clicks_feb,
            (m.clicks_march - f.clicks_feb) AS click_delta,
            DATE_DIFF('day', c.content_updated_date, DATE '2026-03-31') AS days_stale,
            NTILE(4) OVER (ORDER BY DATE_DIFF('day', c.content_updated_date, DATE '2026-03-31')) AS staleness_quartile
        FROM march m
        JOIN feb f USING (client_hash_id, content_hash_id)
        JOIN read_parquet('{DIM_CONTENT}') c
            ON c.content_hash_id = m.content_hash_id
        WHERE c.is_published IS TRUE AND c.is_deleted IS FALSE
          AND c.content_updated_date <= DATE '2026-03-31'   -- clamp: no future edits into a March decision
    )
    SELECT
        staleness_quartile,
        COUNT(*) AS n,
        ROUND(AVG(days_stale), 1) AS avg_days_stale,
        ROUND(AVG(click_delta), 1) AS avg_click_delta_feb_to_march
    FROM joined
    GROUP BY staleness_quartile
    ORDER BY staleness_quartile
""").df()
print(staleness.to_string())

                   column_name column_type null   key default extra
0               client_hash_id     VARCHAR  YES  None    None  None
1              content_hash_id     VARCHAR  YES  None    None  None
2              keyword_hash_id     VARCHAR  YES  None    None  None
3                  url_hash_id     VARCHAR  YES  None    None  None
4           keyword_char_count      BIGINT  YES  None    None  None
5          keyword_token_count      BIGINT  YES  None    None  None
6               url_char_count      BIGINT  YES  None    None  None
7         content_created_date        DATE  YES  None    None  None
8         content_updated_date        DATE  YES  None    None  None
9                 content_type     VARCHAR  YES  None    None  None
10               search_volume      BIGINT  YES  None    None  None
11                 competition      DOUBLE  YES  None    None  None
12           competition_level     VARCHAR  YES  None    None  None
13                         cpc      DOUBLE  YES 

In [8]:
ctr_gap = con.sql(f"""
    WITH march AS (
        SELECT
            client_hash_id, content_hash_id,
            SUM(gsc_clicks) AS clicks_march,
            SUM(gsc_impressions) AS impressions_march,
            AVG(gsc_avg_position) AS avg_position_march,
            SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0) AS ctr_march
        FROM read_parquet('{FACT_MARCH}')
        GROUP BY client_hash_id, content_hash_id
    ),
    feb AS (
        SELECT client_hash_id, content_hash_id, SUM(gsc_clicks) AS clicks_feb
        FROM read_parquet('{FACT_FEB}')
        GROUP BY client_hash_id, content_hash_id
    ),
    position_benchmark AS (
        SELECT
            ROUND(avg_position_march) AS position_bucket,
            AVG(ctr_march) AS expected_ctr_for_position
        FROM march
        GROUP BY ROUND(avg_position_march)
    )
    SELECT
        CASE WHEN m.ctr_march < pb.expected_ctr_for_position THEN 'underperforming' ELSE 'meeting_or_above' END AS ctr_bucket,
        COUNT(*) AS n,
        ROUND(AVG(f.clicks_feb), 1) AS avg_clicks_feb,
        ROUND(AVG(m.clicks_march - f.clicks_feb), 1) AS avg_click_delta_feb_to_march
    FROM march m
    JOIN feb f USING (client_hash_id, content_hash_id)
    JOIN position_benchmark pb ON ROUND(m.avg_position_march) = pb.position_bucket
    GROUP BY ctr_bucket
""").df()
print(ctr_gap.to_string())

         ctr_bucket       n  avg_clicks_feb  avg_click_delta_feb_to_march
0  meeting_or_above   23946             9.3                           6.1
1   underperforming  137593             2.6                           0.4


In [9]:
staleness = con.sql(f"""
    WITH march AS (
        SELECT client_hash_id, content_hash_id, SUM(gsc_clicks) AS clicks_march
        FROM read_parquet('{FACT_MARCH}')
        GROUP BY client_hash_id, content_hash_id
    ),
    feb AS (
        SELECT client_hash_id, content_hash_id, SUM(gsc_clicks) AS clicks_feb
        FROM read_parquet('{FACT_FEB}')
        GROUP BY client_hash_id, content_hash_id
    ),
    joined AS (
        SELECT
            m.client_hash_id, m.content_hash_id, m.clicks_march, f.clicks_feb,
            (m.clicks_march - f.clicks_feb) AS click_delta,
            DATE_DIFF('day', c.content_updated_date, DATE '2026-03-31') AS days_stale,
            NTILE(4) OVER (ORDER BY DATE_DIFF('day', c.content_updated_date, DATE '2026-03-31')) AS staleness_quartile
        FROM march m
        JOIN feb f USING (client_hash_id, content_hash_id)
        JOIN read_parquet('{DIM_CONTENT}') c
            ON c.content_hash_id = m.content_hash_id
        WHERE c.is_published IS TRUE AND c.is_deleted IS FALSE
          AND c.content_updated_date <= DATE '2026-03-31'   -- clamp: no future edits allowed into a March decision
    )
    SELECT
        staleness_quartile,
        COUNT(*) AS n,
        ROUND(AVG(days_stale), 1) AS avg_days_stale,
        ROUND(AVG(click_delta), 1) AS avg_click_delta_feb_to_march
    FROM joined
    GROUP BY staleness_quartile
    ORDER BY staleness_quartile
""").df()
print(staleness.to_string())

   staleness_quartile     n  avg_days_stale  avg_click_delta_feb_to_march
0                   1  9305            32.7                           0.2
1                   2  9305            34.0                           0.1
2                   3  9305            34.0                           0.1
3                   4  9304           155.2                           0.4


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write `work/outputs/baseline_action_score.csv`.*

Weights set directly from the Section 1 verdicts: CTR-gap is CONFIRMED with a large effect, so it dominates. Staleness is MIXED-leaning-FALSE with a tiny, non-monotonic effect, so it gets a small weight rather than zero — kept nonzero only because it's the one signal directly tied to the real refresh flag from the session, but it will rarely be the deciding factor for any row. Same leakage clamp and `is_published`/`is_deleted` filter from Section 1 carried through here.

In [10]:
# --- Score, reason code, action label, then write the CSV ---
# Weights set from Section 1 verdicts: CTR-gap CONFIRMED (large effect) dominates;
# staleness MIXED/leaning-FALSE (tiny, non-monotonic effect) gets a small weight.

STALENESS_WEIGHT = 0.15   # low — verdict was MIXED leaning FALSE
CTR_GAP_WEIGHT   = 0.85   # high — verdict was CONFIRMED, large effect

queue = con.sql(f"""
    WITH march AS (
        SELECT
            client_hash_id, content_hash_id,
            SUM(gsc_clicks) AS clicks_march,
            SUM(gsc_impressions) AS impressions_march,
            AVG(gsc_avg_position) AS avg_position_march,
            SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0) AS ctr_march
        FROM read_parquet('{FACT_MARCH}')
        GROUP BY client_hash_id, content_hash_id
    ),
    with_content AS (
        SELECT
            m.*,
            DATE_DIFF('day', c.content_updated_date, DATE '2026-03-31') AS days_stale
        FROM march m
        JOIN read_parquet('{DIM_CONTENT}') c ON c.content_hash_id = m.content_hash_id
        WHERE c.is_published IS TRUE AND c.is_deleted IS FALSE
          AND c.content_updated_date <= DATE '2026-03-31'   -- same leakage clamp as Section 1
    ),
    position_benchmark AS (
        SELECT ROUND(avg_position_march) AS position_bucket, AVG(ctr_march) AS expected_ctr
        FROM march GROUP BY ROUND(avg_position_march)
    ),
    scored AS (
        SELECT
            w.client_hash_id,
            w.content_hash_id,
            w.days_stale,
            w.ctr_march,
            pb.expected_ctr,
            GREATEST(pb.expected_ctr - w.ctr_march, 0) AS ctr_gap,
            (w.days_stale - MIN(w.days_stale) OVER ()) / NULLIF(MAX(w.days_stale) OVER () - MIN(w.days_stale) OVER (), 0) AS staleness_norm,
            (GREATEST(pb.expected_ctr - w.ctr_march, 0) - MIN(GREATEST(pb.expected_ctr - w.ctr_march, 0)) OVER ())
                / NULLIF(MAX(GREATEST(pb.expected_ctr - w.ctr_march, 0)) OVER () - MIN(GREATEST(pb.expected_ctr - w.ctr_march, 0)) OVER (), 0) AS ctr_gap_norm
        FROM with_content w
        JOIN position_benchmark pb ON ROUND(w.avg_position_march) = pb.position_bucket
    )
    SELECT
        client_hash_id,
        content_hash_id,
        ROUND({STALENESS_WEIGHT} * COALESCE(staleness_norm, 0) + {CTR_GAP_WEIGHT} * COALESCE(ctr_gap_norm, 0), 4) AS score,
        CASE
            WHEN staleness_norm > 0.5 AND ctr_gap_norm > 0.5 THEN 'STALE_LOW_CTR'
            WHEN staleness_norm > 0.5 THEN 'STALE_ONLY'
            WHEN ctr_gap_norm > 0.5 THEN 'CTR_GAP_ONLY'
            ELSE 'HEALTHY'
        END AS reason_code
    FROM scored
""").df()

def action_label(row):
    if row["reason_code"] == "HEALTHY":
        return "PROTECT"
    if row["score"] >= 0.6 and row["reason_code"] in ("STALE_LOW_CTR", "STALE_ONLY"):
        return "REFRESH"
    if row["score"] >= 0.6 and row["reason_code"] == "CTR_GAP_ONLY":
        return "CTR_FIX"
    return "MONITOR"

queue["action"] = queue.apply(action_label, axis=1)
queue = queue.sort_values("score", ascending=False).reset_index(drop=True)

out_path = PROJECT_ROOT / "work" / "outputs" / "baseline_action_score.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)
queue.to_csv(out_path, index=False)
print(f"Wrote {len(queue)} rows to {out_path}")

# Sanity check: reason_code distribution should show CTR_GAP_ONLY as dominant,
# STALE_ONLY / STALE_LOW_CTR should be rare given the Section 1 verdicts.
print(queue["reason_code"].value_counts())
queue.head(10)


Wrote 27801 rows to C:\Projects AI ML\flyrank-ml-starter\work\outputs\baseline_action_score.csv
reason_code
HEALTHY          26968
CTR_GAP_ONLY       575
STALE_ONLY         232
STALE_LOW_CTR       26
Name: count, dtype: int64


,client_hash_id,content_hash_id,score,reason_code,action
0,client_73cda7b4e4f265ea,content_df7c2d2d9bf00c68,0.9696,STALE_LOW_CTR,REFRESH
1,client_73cda7b4e4f265ea,content_92d29c1888906c02,0.9696,STALE_LOW_CTR,REFRESH
2,client_73cda7b4e4f265ea,content_15ae9e160a6f87ae,0.9696,STALE_LOW_CTR,REFRESH
3,client_73cda7b4e4f265ea,content_a7fc7801da9e3b89,0.9655,STALE_LOW_CTR,REFRESH
4,client_73cda7b4e4f265ea,content_f38a9fbbb968f979,0.9655,STALE_LOW_CTR,REFRESH
5,client_73cda7b4e4f265ea,content_ac899d2007985ddb,0.9655,STALE_LOW_CTR,REFRESH
6,client_73cda7b4e4f265ea,content_8eb75774993ebe91,0.9655,STALE_LOW_CTR,REFRESH
7,client_73cda7b4e4f265ea,content_15ca9b3fac65dcf9,0.9655,STALE_LOW_CTR,REFRESH
8,client_73cda7b4e4f265ea,content_58eec996cd0c3b8d,0.9655,STALE_LOW_CTR,REFRESH
9,client_73cda7b4e4f265ea,content_f5d21c17bab789cb,0.9655,STALE_LOW_CTR,REFRESH


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Fill this in after you've actually seen the real top rows — the lines below are a **template for how to write each one**, not real content. Replace with your actual client/content hashes and real numbers once the queue above runs.

| # | action | reason_code | why it's here (1 line) | what would make it wrong |
|---|--------|-------------|--------------------------|----------------------------|
| 1 | REFRESH | STALE_LOW_CTR | 400+ days stale, CTR ~40% below expected for its position | if `days_stale` is wrong because `last_modified_date` reflects a CMS re-save, not real content edit |
| 2 | ... | ... | ... | ... |

*(Continue for all 20. Ten is the assignment minimum per the card; the skeleton header asks for twenty — do twenty if you have time, ten is the floor.)*

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.



# Pull the real top 20 to review by hand above
top20 = queue.head(20)
top20



,client_hash_id,content_hash_id,score,reason_code,action
0,client_73cda7b4e4f265ea,content_df7c2d2d9bf00c68,0.9696,STALE_LOW_CTR,REFRESH
1,client_73cda7b4e4f265ea,content_92d29c1888906c02,0.9696,STALE_LOW_CTR,REFRESH
2,client_73cda7b4e4f265ea,content_15ae9e160a6f87ae,0.9696,STALE_LOW_CTR,REFRESH
3,client_73cda7b4e4f265ea,content_a7fc7801da9e3b89,0.9655,STALE_LOW_CTR,REFRESH
4,client_73cda7b4e4f265ea,content_f38a9fbbb968f979,0.9655,STALE_LOW_CTR,REFRESH
5,client_73cda7b4e4f265ea,content_ac899d2007985ddb,0.9655,STALE_LOW_CTR,REFRESH
6,client_73cda7b4e4f265ea,content_8eb75774993ebe91,0.9655,STALE_LOW_CTR,REFRESH
7,client_73cda7b4e4f265ea,content_15ca9b3fac65dcf9,0.9655,STALE_LOW_CTR,REFRESH
8,client_73cda7b4e4f265ea,content_58eec996cd0c3b8d,0.9655,STALE_LOW_CTR,REFRESH
9,client_73cda7b4e4f265ea,content_f5d21c17bab789cb,0.9655,STALE_LOW_CTR,REFRESH


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Leakage check — this actually caught something real, not just a formality:**
The first version of the staleness query (no clamp on `content_updated_date`) produced *negative* `avg_days_stale` values. That's because `dim_content.parquet` is a single unpartitioned snapshot taken at the dataset's July 2026 export date — most rows' `content_updated_date` falls after March 2026, meaning the unclamped query was silently using knowledge of edits that hadn't happened yet as of the March decision point. Fixed by adding `AND c.content_updated_date <= DATE '2026-03-31'` to every query that joins `dim_content`. This is the exact "no future-window inputs" failure mode the card warns about, caught by noticing the numbers didn't make sense rather than by inspection of the schema alone.

Side effect of the fix, worth naming as its own limitation: the clamp drops the usable row count for the staleness signal by ~87% (73,502 → ~9,305 per quartile). Only a small slice of `dim_content` has update history that's actually knowable as of March — most of what looks like "freshness" in the raw table is really "this got touched sometime in the four months after our decision point," which isn't a fact we're allowed to use.

**Weak picks — fill these in from the real top 20 once reviewed, format below:**
- *(example placeholder, replace with real rows)* a `REFRESH`-labeled page with very low impressions where the CTR gap is likely just noise, not a real signal — worth a minimum-impressions floor before trusting the CTR-gap score.
- *(example placeholder)* any page where `days_stale` sits right at the edge of the clamp (near March 31) — small measurement error there could flip its staleness_norm meaningfully given how few rows survive the clamp.
- *(example placeholder)* a `PROTECT`-labeled page that's actually fine only because it fell in the tiny post-clamp staleness sample, not because staleness was meaningfully assessed.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.




# --- Mechanical leakage check: confirm no later month or unclamped content date reached the queue ---

print("FACT_MARCH points to:", FACT_MARCH)
print("FACT_FEB points to:", FACT_FEB)
print("No fact_content_daily_performance path other than 2026-02 / 2026-03 was constructed in this notebook.")

# Confirm the clamp is actually active: max days_stale should correspond to
# content_updated_date no later than 2026-03-31 for every row in the queue build.
clamp_check = con.sql(f"""
    SELECT MAX(c.content_updated_date) AS latest_update_used
    FROM read_parquet('{DIM_CONTENT}') c
    JOIN read_parquet('{FACT_MARCH}') m ON m.content_hash_id = c.content_hash_id
    WHERE c.is_published IS TRUE AND c.is_deleted IS FALSE
      AND c.content_updated_date <= DATE '2026-03-31'
""").df()
print(clamp_check)
# latest_update_used must be <= 2026-03-31. If it isn't, the clamp isn't working — stop and fix before submitting.

# TODO: replace the placeholder bullets in the markdown cell above with your
# actual 2-3 weak-pick rows once you've eyeballed the real top 20 from queue.head(20).



FACT_MARCH points to: hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet
FACT_FEB points to: hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/*.parquet
No fact_content_daily_performance path other than 2026-02 / 2026-03 was constructed in this notebook.
  latest_update_used
0         2026-03-24


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.